# NHL-Beyond-27 · Book 3 — Corsi Composites & Spicy (z) Analysis

Repo: `ewnike/NHL-Beyond-27` — *MADS Milestone I Project*
Python: 3.13.7 (pyenv env: `nhl_beyond27-3.13.7`)
Editors/Tools: VSCode, Git/GitHub, Postgres + pgAdmin
Logs: `log_utils.py` → `logs/`

**This notebook covers:**

1. Load analysis view/table
2. Define roles (Defense vs Forwards)
3. **Part A — Composite Corsi on z-scores**: unweighted vs role-weighted (D/F) → visuals → regression
4. **Part B — Spicy (within-player z)**: definition → visuals → regression
5. **Part C — Spicy-Weighted (within-player z)**: definition → visuals → regression
6. (Optional) Export small summary CSVs for the paper

**Rel-age order used throughout:** −2, −1, 0, 1, 2 (peak = 0).
We keep **composite Corsi on z-scores** (Part A) separate from **Spicy** metrics (Parts B/C).



In [44]:
import sys

import numpy as np
import pandas as pd

# Optional: these are needed later; harmless to import now
import plotly.express
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf

print(
    "py",
    sys.version.split()[0],
    "| numpy",
    np.__version__,
    "| pandas",
    pd.__version__,
    "| plotly",
    plotly.__version__,
    "| statsmodels",
    sm.__version__,
)

py 3.13.7 | numpy 2.3.2 | pandas 2.3.2 | plotly 6.3.0 | statsmodels 0.14.5


In [45]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm


# DB fallback loader
def load_z_table():
    """
    Try DB first (player_five_year_aligned_z), else CSV fallback at data/outputs/.
    """
    try:
        from db_utils import get_db_engine

        eng = get_db_engine()
        df = pd.read_sql("SELECT * FROM public.player_five_year_aligned_z", eng)
        print("Loaded z-table from DB:", len(df))
        return df
    except Exception as e:
        print("[Info] DB not available or table missing; using CSV fallback:", e)
        csv = Path("data/outputs/player_five_year_aligned_z.csv")
        assert csv.exists(), "CSV fallback not found: data/outputs/player_five_year_aligned_z.csv"
        df = pd.read_csv(csv)
        print("Loaded z-table from CSV:", len(df))
        return df


def load_peak_table():
    """
    EH peak season table with raw CF%/CF60/CA60 (used in Part A).
    DB optional; otherwise read data/peak_player_season_stats.csv.
    """
    try:
        from db_utils import get_db_engine

        eng = get_db_engine()
        q = (
            'SELECT player, season, position, "CF%", "CF/60", "CA/60" '
            "FROM public.player_peak_season"
        )
        df = pd.read_sql(q, eng)
        print("Loaded peak table from DB:", len(df))
        return df
    except Exception as e:
        print("[Info] DB not available; using CSV fallback:", e)
        csv = Path("data/peak_player_season_stats.csv")
        assert csv.exists(), "CSV fallback not found: data/peak_player_season_stats.csv"
        df = pd.read_csv(csv)
        print("Loaded peak table from CSV:", len(df))
        return df


# --- normalize column names to lowercase, no spaces ---
def _norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


# >>> FIX: actually load the data <<<
df_z = _norm_cols(load_z_table())
df_raw = _norm_cols(load_peak_table())

# If the CSV uses 'pos', make it 'position'
alias_map = {"pos": "position"}
df_z.rename(columns={k: v for k, v in alias_map.items() if k in df_z.columns}, inplace=True)
df_raw.rename(columns={k: v for k, v in alias_map.items() if k in df_raw.columns}, inplace=True)


# --- role mapping ---
def to_role(x: str) -> str:
    return "D" if str(x).upper().startswith("D") else "F"


if "position" not in df_z.columns:
    raise KeyError("Expected 'position' in z-table; got columns: " + ", ".join(df_z.columns))
if "position" not in df_raw.columns:
    raise KeyError(
        "Expected 'position' in peak/RAW table; got columns: " + ", ".join(df_raw.columns)
    )

df_z["role"] = df_z["position"].map(to_role)
df_raw["role"] = df_raw["position"].map(to_role)

# enforce rel_age ordering for z table
order = [-2, -1, 0, 1, 2]
if "rel_age" in df_z.columns:
    df_z["rel_age"] = pd.Categorical(
        pd.to_numeric(df_z["rel_age"], errors="coerce"), categories=order, ordered=True
    )

print("Ready. (z cols):", list(df_z.columns)[:12], "…")
print("Ready. (raw cols):", list(df_raw.columns)[:12], "…")

2025-09-30 09:30:59,116 - INFO - db_utils - Using DATABASE_URL from environment.
2025-09-30 09:30:59,161 - INFO - db_utils - Using DATABASE_URL from environment.


Loaded z-table from DB: 1410
[Info] DB not available; using CSV fallback: sqlalchemy.cyextension.immutabledict.immutabledict is not a sequence
Loaded peak table from CSV: 3121
Ready. (z cols): ['player', 'position', 'peak_year', 'rel_age', 'start_year', 'season', 'age', 'cf_pct', 'cf60', 'ca60', 'cf_pct_z', 'cf60_z'] …
Ready. (raw cols): ['player', 'eh_id', 'api id', 'season', 'team', 'position', 'shoots', 'birthday', 'age', 'draft yr', 'draft rd', 'draft ov'] …


**## Part A — Composite Corsi on Z-scores (Unweighted vs Role-Weighted)**



**Goal.** Summarize play-driving with a single standardized composite, built from **within-player z-scores** (each metric centered/scaled by that player’s 5-year baseline):

* `cf_pct_z` — possession share (CF%)
* `cf60_z` — shot creation per 60 (CF/60)
* `ca60_z` — shot suppression per 60 (CA/60) *(enters with a minus sign)*

### Two composites

**1) Unweighted composite z**

$$
\text{z_comp_unweighted} ;=; \mathrm{mean}\big(cf_pct_z,; cf60_z,; -,ca60_z\big)
$$

**2) Role-weighted composite z** *(simple, fixed heuristics for this milestone)*

* **Defense (D):** emphasize suppression a bit more; creation a bit less
  $$
  \text{z_comp_weighted} ;=; 0.5,cf_pct_z ;+; 0.2,cf60_z ;-; 0.3,ca60_z
  $$

* **Forwards (F):** emphasize creation a bit more; suppression a bit less
  $$
  \text{z_comp_weighted} ;=; 0.5,cf_pct_z ;+; 0.3,cf60_z ;-; 0.2,ca60_z
  $$

> **Why z-scores?** They avoid raw-scale mixing and make the composite interpretable as “high/low **relative to the same player’s baseline**.”
> **Why these weights?** Clear, documented heuristics to keep Book 3 simple. You can tune them later with data-driven optimization or cross-validation.

### What to look for in the plots

* **By `rel_age` (−2, −1, 0, +1, +2):** trajectories around peak (0) for Defense vs Forwards.
* **Unweighted vs role-weighted:** how weighting shifts the relative separation of roles.
* **CI ribbons:** 95% confidence intervals around the mean composite by role × `rel_age`.

### Outputs in this part

* Line charts of **mean composite z** by role across `rel_age` with **95% CI** ribbons.
* Histograms / boxplots comparing the distribution of unweighted vs role-weighted composites by role.
* (Optional) Export of summary tables used to render the figures.



In [46]:
import numpy as np
import pandas as pd

dfz = df_z.copy()

# Role from position
dfz["role"] = np.where(dfz["position"].astype(str).str.upper().str.startswith("D"), "D", "F")


# Helper: mean of available values
def _mean_available(vals):
    v = [x for x in vals if pd.notna(x)]
    return float(np.mean(v)) if v else np.nan


# Unweighted composite on z-scores
def comp_unw(r):
    return _mean_available([r["cf_pct_z"], r["cf60_z"], -r["ca60_z"]])


# Role-weighted composite on z-scores
W_D = {"cf_pct_z": 0.5, "cf60_z": 0.2, "ca60_z": 0.3}
W_F = {"cf_pct_z": 0.5, "cf60_z": 0.3, "ca60_z": 0.2}


def comp_w(r):
    w = W_D if r["role"] == "D" else W_F
    num, den = 0.0, 0.0
    for k, wt in w.items():
        v = r.get(k)
        if pd.notna(v):
            if k == "ca60_z":
                v = -v  # suppress against
            num += wt * v
            den += wt
    return num / den if den else np.nan


dfz["z_comp_unweighted"] = dfz.apply(comp_unw, axis=1)
dfz["z_comp_weighted"] = dfz.apply(comp_w, axis=1)

# Keep rel_age ordered for plots
order = [-2, -1, 0, 1, 2]
if "rel_age" in dfz.columns:
    dfz["rel_age"] = pd.Categorical(
        pd.to_numeric(dfz["rel_age"], errors="coerce"), categories=order, ordered=True
    )

dfz[["player", "season", "role", "rel_age", "z_comp_unweighted", "z_comp_weighted"]].head(8)

,player,season,role,rel_age,z_comp_unweighted,z_comp_weighted
0,Adam Henrique,16-17,F,-1,-0.405994,-0.428285
1,Adam Henrique,19-20,F,2,0.778992,0.931875
2,Adam Henrique,15-16,F,-2,-0.232633,-0.538525
3,Adam Henrique,17-18,F,0,0.520218,0.683384
4,Adam Henrique,18-19,F,1,-0.660584,-0.648449
5,Adam Larsson,20-21,D,0,-0.557387,-0.552449
6,Adam Larsson,22-23,D,2,1.471390,1.507333
7,Adam Larsson,21-22,D,1,0.027213,0.044183


In [47]:
import numpy as np
import pandas as pd
from plotly import colors as pcolors


def hex_to_rgba(hex_color: str, alpha: float = 0.2) -> str:
    """Convert '#RRGGBB' to 'rgba(r,g,b,alpha)'."""
    h = hex_color.lstrip("#")
    if len(h) != 6:
        raise ValueError(f"Expected 6-digit hex like '#RRGGBB', got {hex_color}")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def mean_ci(df: pd.DataFrame, col: str) -> pd.DataFrame:
    if not {"role", "rel_age", col}.issubset(df.columns):
        missing = {"role", "rel_age", col} - set(df.columns)
        raise KeyError(f"Column(s) missing from df: {missing}")

    g = (
        df.groupby(["role", "rel_age"])
        .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
        .reset_index()
    )
    # handle n==1 (sd is NaN) safely
    g["se"] = g["sd"].fillna(0) / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g


def ribbon_line(means: pd.DataFrame, title: str, ylab: str) -> go.Figure:
    fig = go.Figure()

    # Default palette with fallbacks for unseen roles
    base_palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    fallback_cycle = pcolors.qualitative.Plotly  # a list of hex colors
    cycle_idx = 0

    # ensure rel_age is numeric for sorting; keep original for axis categories
    rel_age_order = [-2, -1, 0, 1, 2]
    means = means.copy()
    means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")

    # map roles to colors with stable fallback
    role_colors = {}
    for role in means["role"].dropna().unique():
        if role in base_palette:
            role_colors[role] = base_palette[role]
        else:
            role_colors[role] = fallback_cycle[cycle_idx % len(fallback_cycle)]
            cycle_idx += 1

    for role in means["role"].dropna().unique():
        sub = means.loc[means["role"] == role].sort_values("rel_age")
        sub = sub[sub["rel_age"].isin(rel_age_order)]  # keep known x
        if sub.empty:
            continue

        # use strings for categorical x so categoryarray works
        x = sub["rel_age"].astype(int).astype(str)
        c = role_colors[role]

        # Upper bound (invisible line to anchor fill)
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["upper"],
                line=dict(width=0),
                hoverinfo="skip",
                showlegend=False,
                name=f"{role} 95% CI (upper)",
            )
        )

        # Lower bound + fill between lower and the previous (upper) trace
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["lower"],
                line=dict(width=0),
                hoverinfo="skip",
                fill="tonexty",
                fillcolor=hex_to_rgba(c, alpha=0.2),
                showlegend=False,
                name=f"{role} 95% CI",
            )
        )

        # Mean line
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=c, width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    # y-range (guard against all-NaN)
    lower_min = np.nanmin(means["lower"].to_numpy()) if "lower" in means else np.nan
    upper_max = np.nanmax(means["upper"].to_numpy()) if "upper" in means else np.nan
    if np.isfinite(lower_min) and np.isfinite(upper_max):
        y_min = float(np.floor(lower_min - 0.2))
        y_max = float(np.ceil(upper_max + 0.2))
        y_range = [y_min, y_max]
    else:
        y_range = None  # let Plotly autoscale

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2", "-1", "0", "1", "2"])
    if y_range:
        fig.update_yaxes(range=y_range, dtick=0.2)
    else:
        fig.update_yaxes(dtick=0.2)

    return fig


# ---- Example usage (make sure dfz exists and has the right columns) ----
# m_unw = mean_ci(dfz, "z_comp_unweighted")
# m_w   = mean_ci(dfz, "z_comp_weighted")
# ribbon_line(m_unw, "Composite z (Unweighted) — Mean by rel_age (95% CI)", "unweighted z").show()
# ribbon_line(m_w,   "Composite z (Role-weighted) — Mean by rel_age (95% CI)", "role-weighted z").show()

In [48]:
# Unweighted
fig_h1 = px.histogram(
    dfz,
    x="z_comp_unweighted",
    color="role",
    barmode="overlay",
    nbins=40,
    opacity=0.6,
    title="Composite z (Unweighted) — Distribution by Role",
    labels={"z_comp_unweighted": "unweighted composite z"},
)
fig_h1.show()

# Weighted
fig_h2 = px.histogram(
    dfz,
    x="z_comp_weighted",
    color="role",
    barmode="overlay",
    nbins=40,
    opacity=0.6,
    title="Composite z (Role-weighted) — Distribution by Role",
    labels={"z_comp_weighted": "role-weighted composite z"},
)
fig_h2.show()

In [49]:
# Unweighted
fig_b1 = px.box(
    dfz,
    x="role",
    y="z_comp_unweighted",
    title="Composite z (Unweighted) — Boxplot by Role",
    labels={"z_comp_unweighted": "unweighted composite z"},
)
fig_b1.show()

# Weighted
fig_b2 = px.box(
    dfz,
    x="role",
    y="z_comp_weighted",
    title="Composite z (Role-weighted) — Boxplot by Role",
    labels={"z_comp_weighted": "role-weighted composite z"},
)
fig_b2.show()

In [50]:
import numpy as np
import plotly.graph_objects as go


def hex_to_rgba(hex_color: str, alpha: float = 0.2) -> str:
    """Convert '#RRGGBB' to 'rgba(r,g,b,alpha)'."""
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def mean_ci(df, col):
    g = (
        df.groupby(["role", "rel_age"], observed=True)
        .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
        .reset_index()
    )
    g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g


def ribbon_line(means, title, ylab, ytick=0.05, pad=0.05):
    fig = go.Figure()
    palette = {"D": "#1f77b4", "F": "#ff7f0e"}

    means = means.copy()
    if "rel_age" in means.columns:
        means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")

    for role in means["role"].dropna().unique():
        sub = means[means["role"] == role].sort_values("rel_age")
        if sub.empty:
            continue
        x = sub["rel_age"].astype(int).astype(str)
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=palette.get(role, "#888888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    # Compute a tight range around the data (including CI if present)
    lo = np.nanmin(np.column_stack([means.get("lower", means["mean"]).to_numpy()]))
    hi = np.nanmax(np.column_stack([means.get("upper", means["mean"]).to_numpy()]))
    span = float(hi - lo) if np.isfinite(hi - lo) else 0.0
    pad_abs = max(span * pad, 0.02)  # ensure a tiny buffer
    y_range = [float(lo - pad_abs), float(hi + pad_abs)] if np.isfinite(span) else None

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2", "-1", "0", "1", "2"])
    fig.update_yaxes(
        range=y_range,  # tight zoom
        dtick=ytick,  # finer grid: try 0.05 or 0.02
        tickformat=".2f",  # more precise labels
        zeroline=True,  # accent reference
        zerolinewidth=1,
        gridwidth=0.5,
    )
    return fig


# Build tables and plot
m_unw = mean_ci(dfz, "z_comp_unweighted")
m_w = mean_ci(dfz, "z_comp_weighted")

ribbon_line(m_unw, "Composite z (Unweighted) — Mean by rel_age (95% CI)", "unweighted z").show()
ribbon_line(m_w, "Composite z (Role-weighted) — Mean by rel_age (95% CI)", "role-weighted z").show()

regression analysis for corsi composites

In [63]:
# --- OLS: composite z (unweighted & role-weighted) ~ role * rel_age (categorical), HC3 robust SEs

import numpy as np
import pandas as pd

# Work on a copy
zz = dfz.copy()

# Ensure role exists (D vs F) if not already present
if "role" not in zz.columns:
    zz["role"] = np.where(zz["position"].astype(str).str.upper().str.startswith("D"), "D", "F")

# Enforce rel_age categorical ordering if present
order = [-2, -1, 0, 1, 2]
if "rel_age" in zz.columns:
    zz["rel_age"] = pd.Categorical(
        pd.to_numeric(zz["rel_age"], errors="coerce"),
        categories=order,
        ordered=True,
    )


def run_ols(ycol: str, data: pd.DataFrame):
    """
    Fit OLS with HC3 robust SEs.
    If rel_age exists, use interaction: y ~ C(role) * C(rel_age, Treatment(0))
    Otherwise: y ~ C(role)
    Returns (model, tidy_coefs_df).
    """
    if "rel_age" in data.columns and data["rel_age"].notna().any():
        formula = f"{ycol} ~ C(role) * C(rel_age, Treatment(0))"
    else:
        formula = f"{ycol} ~ C(role)"

    model = smf.ols(formula, data=data).fit(cov_type="HC3")
    print(f"\n=== OLS (HC3) for {ycol} ===")
    print(model.summary())

    # Tidy coef table (coef, SE, p, 95% CI) + R^2 for convenience
    coefs = pd.DataFrame(
        {
            "term": model.params.index,
            "coef": model.params.values,
            "se": model.bse.values,
            "p": model.pvalues.values,
        }
    )
    ci = model.conf_int()
    coefs["ci_lo"] = ci[0].values
    coefs["ci_hi"] = ci[1].values
    coefs["r2"] = model.rsquared
    coefs["r2_adj"] = model.rsquared_adj
    return model, coefs


# Run for both composites
m_unw, coefs_unw = run_ols("z_comp_unweighted", zz)
m_w, coefs_w = run_ols("z_comp_weighted", zz)

# Show tidy tables
display(coefs_unw)
display(coefs_w)

# Optional: marginal predictions by role × rel_age (95% CI) for plotting
if "rel_age" in zz.columns:
    grid = pd.MultiIndex.from_product([["D", "F"], order], names=["role", "rel_age"]).to_frame(
        index=False
    )
    pred_unw = m_unw.get_prediction(grid).summary_frame(alpha=0.05)
    pred_w = m_w.get_prediction(grid).summary_frame(alpha=0.05)

    pred_unw = pd.concat([grid, pred_unw[["mean", "mean_ci_lower", "mean_ci_upper"]]], axis=1)
    pred_w = pd.concat([grid, pred_w[["mean", "mean_ci_lower", "mean_ci_upper"]]], axis=1)

    pred_unw.rename(
        columns={"mean": "y", "mean_ci_lower": "ci_lo", "mean_ci_upper": "ci_hi"}, inplace=True
    )
    pred_w.rename(
        columns={"mean": "y", "mean_ci_lower": "ci_lo", "mean_ci_upper": "ci_hi"}, inplace=True
    )

    print("\nPredictions (unweighted):")
    display(pred_unw)

    print("\nPredictions (role-weighted):")
    display(pred_w)


=== OLS (HC3) for z_comp_unweighted ===
                            OLS Regression Results                            
Dep. Variable:      z_comp_unweighted   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     1.816
Date:                Tue, 30 Sep 2025   Prob (F-statistic):             0.0610
Time:                        09:40:10   Log-Likelihood:                -1588.3
No. Observations:                1410   AIC:                             3197.
Df Residuals:                    1400   BIC:                             3249.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------

,term,coef,se,p,ci_lo,ci_hi,r2,r2_adj
0,Intercept,-0.009147,0.067207,0.891746,-0.140870,0.122577,0.012098,0.005747
1,C(role)[T.F],0.009825,0.087905,0.911010,-0.162466,0.182115,0.012098,0.005747
2,"C(rel_age, Treatment(0))[T.-2]",0.075788,0.108040,0.483002,-0.135966,0.287542,0.012098,0.005747
3,"C(rel_age, Treatment(0))[T.-1]",0.069372,0.096981,0.474413,-0.120707,0.259452,0.012098,0.005747
4,"C(rel_age, Treatment(0))[T.1]",-0.008445,0.103823,0.935173,-0.211934,0.195044,0.012098,0.005747
5,"C(rel_age, Treatment(0))[T.2]",-0.090983,0.100895,0.367183,-0.288733,0.106767,0.012098,0.005747
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.068091,0.135083,0.614213,-0.332848,0.196666,0.012098,0.005747
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.053009,0.123868,0.668686,-0.189767,0.295786,0.012098,0.005747
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.039235,0.128253,0.759668,-0.212137,0.290606,0.012098,0.005747
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.073276,0.130671,0.574955,-0.329387,0.182834,0.012098,0.005747


,term,coef,se,p,ci_lo,ci_hi,r2,r2_adj
0,Intercept,-0.013688,0.070520,0.846097,-0.151905,0.124529,0.012146,0.005795
1,C(role)[T.F],0.004032,0.091895,0.965007,-0.176079,0.184142,0.012146,0.005795
2,"C(rel_age, Treatment(0))[T.-2]",0.088182,0.112975,0.435070,-0.133245,0.309609,0.012146,0.005795
3,"C(rel_age, Treatment(0))[T.-1]",0.088050,0.100957,0.383129,-0.109823,0.285923,0.012146,0.005795
4,"C(rel_age, Treatment(0))[T.1]",-0.013579,0.108420,0.900331,-0.226078,0.198920,0.012146,0.005795
5,"C(rel_age, Treatment(0))[T.2]",-0.094213,0.105886,0.373599,-0.301746,0.113321,0.012146,0.005795
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.072322,0.141202,0.608519,-0.349074,0.204429,0.012146,0.005795
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.050575,0.129088,0.695216,-0.202432,0.303582,0.012146,0.005795
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.062279,0.134117,0.642388,-0.200585,0.325143,0.012146,0.005795
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.060689,0.136862,0.657452,-0.328934,0.207555,0.012146,0.005795



Predictions (unweighted):


,role,rel_age,y,ci_lo,ci_hi
0,D,-2,0.066641,-0.099156,0.232439
1,D,-1,0.060226,-0.076811,0.197263
2,D,0,-0.009147,-0.140870,0.122577
3,D,1,-0.017591,-0.172693,0.137511
4,D,2,-0.100130,-0.247622,0.047363
5,F,-2,0.008375,-0.105307,0.122057
6,F,-1,0.123060,0.020702,0.225418
7,F,0,0.000678,-0.110376,0.111732
8,F,1,0.031468,-0.065727,0.128663
9,F,2,-0.163581,-0.282554,-0.044608



Predictions (role-weighted):


,role,rel_age,y,ci_lo,ci_hi
0,D,-2,0.074494,-0.098497,0.247485
1,D,-1,0.074362,-0.067236,0.215959
2,D,0,-0.013688,-0.151905,0.124529
3,D,1,-0.027267,-0.188673,0.134139
4,D,2,-0.107901,-0.262710,0.046909
5,F,-2,0.006203,-0.113068,0.125475
6,F,-1,0.128968,0.021623,0.236313
7,F,0,-0.009656,-0.125137,0.105825
8,F,1,0.039043,-0.063940,0.142027
9,F,2,-0.164558,-0.289255,-0.039862


In [51]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


def mean_ci(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = (
        df.groupby(["role", "rel_age"])
        .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
        .reset_index()
    )
    # sd is NaN when n==1; treat as 0 so CI collapses to mean
    g["se"] = g["sd"].fillna(0) / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g


def line_only(means: pd.DataFrame, title: str, ylab: str) -> go.Figure:
    if means.empty:
        print("[WARN] 'means' is empty.")
    means = means.copy()
    # RELIABLE x: coerce to numeric, drop NaN rel_age rows
    means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")
    means = means.dropna(subset=["rel_age"])

    # quick debug
    print("[DEBUG] roles:", means["role"].unique(), "rel_age:", sorted(means["rel_age"].unique()))

    palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    fig = go.Figure()

    for role in means["role"].dropna().unique():
        sub = means.loc[means["role"] == role].sort_values("rel_age")
        if sub.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=sub["rel_age"],  # numeric axis → less fragile
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=palette.get(role, "#888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(tickvals=[-2, -1, 0, 1, 2])
    fig.update_yaxes(dtick=0.2)
    return fig


def line_with_errorbars(means: pd.DataFrame, title: str, ylab: str) -> go.Figure:
    if means.empty:
        print("[WARN] 'means' is empty.")
    means = means.copy()
    means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")
    means = means.dropna(subset=["rel_age"])
    means["ci_half"] = (means["upper"] - means["mean"]).astype(float)
    means["ci_half"] = means["ci_half"].clip(lower=0).fillna(0)

    print("[DEBUG] roles:", means["role"].unique(), "rel_age:", sorted(means["rel_age"].unique()))

    palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    fig = go.Figure()

    for role in means["role"].dropna().unique():
        sub = means.loc[means["role"] == role].sort_values("rel_age")
        if sub.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=sub["rel_age"],
                y=sub["mean"],
                mode="lines+markers",
                error_y=dict(type="data", array=sub["ci_half"], visible=True),
                line=dict(color=palette.get(role, "#888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(tickvals=[-2, -1, 0, 1, 2])
    fig.update_yaxes(dtick=0.2)
    return fig

In [52]:
def line_only(means, title, ylab):
    fig = go.Figure()
    palette = {"D": "#1f77b4", "F": "#ff7f0e"}

    if "rel_age" in means.columns:
        means = means.copy()
        means["rel_age"] = means["rel_age"].astype(float)

    for role in means["role"].dropna().unique():
        sub = means[means["role"] == role].sort_values("rel_age")
        fig.add_trace(
            go.Scatter(
                x=sub["rel_age"].astype(str),
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=palette.get(role, "#888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2", "-1", "0", "1", "2"])
    fig.update_yaxes(dtick=0.2)
    return fig

** Quick note on negatives **
Your composites are z-scores (within-player). It’s normal for means to sit near 0 and be negative/positive depending on the season relative to the player’s baseline. That’s expected behavior, not an error.

In [53]:
# Part A — Regression on composite z-scores (unweighted & role-weighted)
# Outcome ~ C(role) * C(rel_age, Treatment(0))
# - Intercept is Defense at rel_age=0
# - HC3 robust standard errors (heteroskedasticity-consistent)

import numpy as np
import pandas as pd

# ---- safety: keep a working copy; coerce types / categories
dfA = dfz.copy()

# role: ensure only "D" vs "F"
dfA["role"] = np.where(dfA["role"].astype(str).str.upper().str.startswith("D"), "D", "F")

# rel_age: ordered categorical with desired baseline 0
order = [-2, -1, 0, 1, 2]
dfA["rel_age"] = pd.Categorical(
    pd.to_numeric(dfA["rel_age"], errors="coerce"), categories=order, ordered=True
)


# helper to fit + tidy
def fit_and_tidy(formula: str, data: pd.DataFrame, cov_type: str = "HC3"):
    model = smf.ols(formula, data=data).fit(cov_type=cov_type)
    # tidy table
    ci = model.conf_int(alpha=0.05)
    tidy = pd.DataFrame(
        {
            "term": model.params.index,
            "coef": model.params.values,
            "se": model.bse.values,
            "t": model.tvalues.values,
            "p": model.pvalues.values,
            "ci_lo": ci[0].values,
            "ci_hi": ci[1].values,
        }
    ).reset_index(drop=True)
    return model, tidy


# 1) Unweighted composite
form_unw = "z_comp_unweighted ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"
m_unw, tidy_unw = fit_and_tidy(form_unw, dfA, cov_type="HC3")

# 2) Role-weighted composite
form_w = "z_comp_weighted ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"
m_w, tidy_w = fit_and_tidy(form_w, dfA, cov_type="HC3")

print("Unweighted composite — OLS with HC3 SEs")
display(tidy_unw)

print("Role-weighted composite — OLS with HC3 SEs")
display(tidy_w)

Unweighted composite — OLS with HC3 SEs


,term,coef,se,t,p,ci_lo,ci_hi
0,Intercept,-0.009147,0.067207,-0.136095,0.891746,-0.140870,0.122577
1,"C(role, Treatment('D'))[T.F]",0.009825,0.087905,0.111764,0.911010,-0.162466,0.182115
2,"C(rel_age, Treatment(0))[T.-2]",0.075788,0.108040,0.701482,0.483002,-0.135966,0.287542
3,"C(rel_age, Treatment(0))[T.-1]",0.069372,0.096981,0.715317,0.474413,-0.120707,0.259452
4,"C(rel_age, Treatment(0))[T.1]",-0.008445,0.103823,-0.081338,0.935173,-0.211934,0.195044
5,"C(rel_age, Treatment(0))[T.2]",-0.090983,0.100895,-0.901762,0.367183,-0.288733,0.106767
6,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",-0.068091,0.135083,-0.504069,0.614213,-0.332848,0.196666
7,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",0.053009,0.123868,0.427952,0.668686,-0.189767,0.295786
8,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",0.039235,0.128253,0.305916,0.759668,-0.212137,0.290606
9,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",-0.073276,0.130671,-0.560770,0.574955,-0.329387,0.182834


Role-weighted composite — OLS with HC3 SEs


,term,coef,se,t,p,ci_lo,ci_hi
0,Intercept,-0.013688,0.070520,-0.194101,0.846097,-0.151905,0.124529
1,"C(role, Treatment('D'))[T.F]",0.004032,0.091895,0.043872,0.965007,-0.176079,0.184142
2,"C(rel_age, Treatment(0))[T.-2]",0.088182,0.112975,0.780545,0.435070,-0.133245,0.309609
3,"C(rel_age, Treatment(0))[T.-1]",0.088050,0.100957,0.872145,0.383129,-0.109823,0.285923
4,"C(rel_age, Treatment(0))[T.1]",-0.013579,0.108420,-0.125244,0.900331,-0.226078,0.198920
5,"C(rel_age, Treatment(0))[T.2]",-0.094213,0.105886,-0.889752,0.373599,-0.301746,0.113321
6,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",-0.072322,0.141202,-0.512188,0.608519,-0.349074,0.204429
7,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",0.050575,0.129088,0.391787,0.695216,-0.202432,0.303582
8,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",0.062279,0.134117,0.464362,0.642388,-0.200585,0.325143
9,"C(role, Treatment('D'))[T.F]:C(rel_age, Treatm...",-0.060689,0.136862,-0.443434,0.657452,-0.328934,0.207555


## Generate adjusted means (predictions by role and rel_age + 95% CIs and plot)

In [54]:
# # Compute asymmetric error bars (no ribbons)
def add_ci_errors(df):
    d = df.copy()
    d["err_plus"] = d["ci_hi"] - d["yhat"]
    d["err_minus"] = d["yhat"] - d["ci_lo"]
    return d


pred_unw_e = add_ci_errors(pred_unw)
pred_w_e = add_ci_errors(pred_w)

# Lines + markers + 95% CI error bars (no fill)
fig_unw = px.line(
    pred_unw_e,
    x="rel_age",
    y="yhat",
    color="role",
    markers=True,
    category_orders={"rel_age": order},
    labels={"yhat": "Adjusted mean (unweighted z)"},
    title="Composite z (Unweighted) — Adjusted Means by rel_age (95% CI)",
)
fig_unw.update_traces(
    error_y=dict(array=pred_unw_e["err_plus"], arrayminus=pred_unw_e["err_minus"], visible=True),
    selector=dict(mode="lines+markers"),
)
fig_unw.update_yaxes(tickformat=".2f", dtick=0.05)
fig_unw.show()

fig_w = px.line(
    pred_w_e,
    x="rel_age",
    y="yhat",
    color="role",
    markers=True,
    category_orders={"rel_age": order},
    labels={"yhat": "Adjusted mean (role-weighted z)"},
    title="Composite z (Role-weighted) — Adjusted Means by rel_age (95% CI)",
)
fig_w.update_traces(
    error_y=dict(array=pred_w_e["err_plus"], arrayminus=pred_w_e["err_minus"], visible=True),
    selector=dict(mode="lines+markers"),
)
fig_w.update_yaxes(tickformat=".2f", dtick=0.05)
fig_w.show()

**Export table information to data/outputs

In [55]:
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

tidy_unw.to_csv(OUT / "reg_partA_unweighted_tidy.csv", index=False)
tidy_w.to_csv(OUT / "reg_partA_weighted_tidy.csv", index=False)
pred_unw.to_csv(OUT / "reg_partA_unweighted_preds.csv", index=False)
pred_w.to_csv(OUT / "reg_partA_weighted_preds.csv", index=False)

print(
    "Wrote:",
    OUT / "reg_partA_unweighted_tidy.csv",
    ",",
    OUT / "reg_partA_weighted_tidy.csv",
    ",",
    OUT / "reg_partA_unweighted_preds.csv",
    ",",
    OUT / "reg_partA_weighted_preds.csv",
)

Wrote: data/outputs/reg_partA_unweighted_tidy.csv , data/outputs/reg_partA_weighted_tidy.csv , data/outputs/reg_partA_unweighted_preds.csv , data/outputs/reg_partA_weighted_preds.csv


## Part B — Spicy (within-player z)

**What is it?**
A composite built from within-player standardized components; it answers:  
**“Relative to this same player’s five-year baseline, how hot/cold is this season?”**

This section uses the **z-table** we built earlier (`player_five_year_aligned_z`).  
We’ll explore **spicy (unweighted)** on its own.


In [56]:
z = df_z.copy()

# Distribution by role
fig = px.histogram(
    z,
    x="spicy_score",
    color="role",
    barmode="overlay",
    nbins=40,
    opacity=0.6,
    title="Spicy (z) — Distribution by Role",
)
fig.show()

# Mean spicy by rel_age × role
if "rel_age" in z.columns:
    mean_spicy = z.groupby(["role", "rel_age"], as_index=False)["spicy_score"].mean(
        numeric_only=True
    )
    fig2 = px.line(
        mean_spicy,
        x="rel_age",
        y="spicy_score",
        color="role",
        markers=True,
        title="Spicy (z) — Mean by Role × rel_age",
    )
    fig2.update_xaxes(categoryorder="array", categoryarray=[-2, -1, 0, 1, 2])
    fig2.show()

z[["player", "season", "role", "rel_age", "spicy_score"]].head(8)

/var/folders/hb/kbyr_0y166n0nvnd_cqyjzth0000gn/T/ipykernel_43350/2992377503.py:10: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,player,season,role,rel_age,spicy_score
0,Adam Henrique,16-17,F,-1,-0.405994
1,Adam Henrique,19-20,F,2,0.778992
2,Adam Henrique,15-16,F,-2,-0.232633
3,Adam Henrique,17-18,F,0,0.520218
4,Adam Henrique,18-19,F,1,-0.660584
5,Adam Larsson,20-21,D,0,-0.557387
6,Adam Larsson,22-23,D,2,1.471390
7,Adam Larsson,21-22,D,1,0.027213


In [57]:
# OLS: spicy ~ role * rel_age (categorical), HC3 robust SEs
if "rel_age" in z.columns:
    m_spicy = smf.ols("spicy_score ~ C(role) * C(rel_age, Treatment(0))", data=z).fit(
        cov_type="HC3"
    )
else:
    m_spicy = smf.ols("spicy_score ~ C(role)", data=z).fit(cov_type="HC3")

print(m_spicy.summary())

# Tidy
coefs_spicy = pd.DataFrame(
    {
        "term": m_spicy.params.index,
        "coef": m_spicy.params.values,
        "se": m_spicy.bse.values,
        "p": m_spicy.pvalues.values,
    }
)
ci = m_spicy.conf_int()
coefs_spicy["ci_lo"] = ci[0].values
coefs_spicy["ci_hi"] = ci[1].values
coefs_spicy

                            OLS Regression Results                            
Dep. Variable:            spicy_score   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     1.816
Date:                Tue, 30 Sep 2025   Prob (F-statistic):             0.0610
Time:                        09:30:59   Log-Likelihood:                -1588.3
No. Observations:                1410   AIC:                             3197.
Df Residuals:                    1400   BIC:                             3249.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

,term,coef,se,p,ci_lo,ci_hi
0,Intercept,-0.009147,0.067207,0.891746,-0.140870,0.122577
1,C(role)[T.F],0.009825,0.087905,0.911010,-0.162466,0.182115
2,"C(rel_age, Treatment(0))[T.-2]",0.075788,0.108040,0.483002,-0.135966,0.287542
3,"C(rel_age, Treatment(0))[T.-1]",0.069372,0.096981,0.474413,-0.120707,0.259452
4,"C(rel_age, Treatment(0))[T.1]",-0.008445,0.103823,0.935173,-0.211934,0.195044
5,"C(rel_age, Treatment(0))[T.2]",-0.090983,0.100895,0.367183,-0.288733,0.106767
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.068091,0.135083,0.614213,-0.332848,0.196666
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.053009,0.123868,0.668686,-0.189767,0.295786
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.039235,0.128253,0.759668,-0.212137,0.290606
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.073276,0.130671,0.574955,-0.329387,0.182834


## Part C — Spicy-Weighted (within-player z, role-aware)

This is the role-aware composite **in z-space** (already computed in SQL as `spicy_weighted`).  
Interpretation remains within-player: positive = hotter than that player’s own baseline, negative = colder.


In [58]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# --- start with your table ---
zw = dfz.copy()

# ensure role as D/F
role_col = "position" if "position" in zw.columns else "role"
zw["role"] = zw[role_col].astype(str).str.upper().str.startswith("D").map({True: "D", False: "F"})

# keep useful columns if they exist (DON'T drop others)
keep_maybe = ["spicy_score", "spicy_weighted", "role", "rel_age", "player", "season"]
zw = zw[[c for c in keep_maybe if c in zw.columns]].copy()

# -------- A) SCATTER: spicy (x) vs spicy_weighted (y), faceted by role --------
sc = zw.dropna(
    subset=[c for c in ["spicy_score", "spicy_weighted", "role"] if c in zw.columns]
).copy()

mn = float(np.nanmin([sc["spicy_score"].min(), sc["spicy_weighted"].min()]))
mx = float(np.nanmax([sc["spicy_score"].max(), sc["spicy_weighted"].max()]))
pad = (mx - mn) * 0.05 if np.isfinite(mx - mn) else 0.1
rng = [mn - pad, mx + pad]

fig = px.scatter(
    sc,
    x="spicy_score",
    y="spicy_weighted",
    facet_row="role",
    color="role",
    opacity=0.6,
    trendline="ols",
    trendline_scope="trace",
    labels={"spicy_score": "SPICY (equal-weight)", "spicy_weighted": "SPICY (role-weighted)"},
    title="Within-role comparison: SPICY vs SPICY (role-weighted)",
)

# add y=x line to each facet + shared axes
for r in ["D", "F"]:
    fig.add_trace(
        go.Scatter(
            x=rng,
            y=rng,
            mode="lines",
            line=dict(width=1, dash="dash"),
            name="y = x",
            showlegend=(r == "D"),
        ),
        row=1 if r == "D" else 2,
        col=1,
    )
fig.update_xaxes(matches="x", range=rng, tickformat=".2f")
fig.update_yaxes(matches="y", range=rng, tickformat=".2f")
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("role=", "")))
fig.update_layout(
    template="plotly_white", legend_title_text="Role", margin=dict(l=60, r=10, t=60, b=40)
)
fig.show()

# -------- B) LINES: mean spicy_weighted by rel_age × role (only if rel_age exists) --------
if "rel_age" in zw.columns:
    # normalize rel_age to ordered set
    order = [-2, -1, 0, 1, 2]
    zw["rel_age"] = pd.to_numeric(zw["rel_age"], errors="coerce")
    zw = zw[zw["rel_age"].isin(order)].copy()
    zw["rel_age"] = pd.Categorical(zw["rel_age"], categories=order, ordered=True)

    g = (
        zw.groupby(["role", "rel_age"], observed=True)["spicy_weighted"]
        .mean(numeric_only=True)
        .reset_index()
    )
    fig_lines = px.line(
        g,
        x="rel_age",
        y="spicy_weighted",
        color="role",
        markers=True,
        title="Spicy-Weighted (z) — Mean by Role × rel_age",
        category_orders={"rel_age": order, "role": ["D", "F"]},
    )
    fig_lines.update_xaxes(categoryorder="array", categoryarray=order)
    fig_lines.update_yaxes(tickformat=".2f", dtick=0.05)
    fig_lines.update_layout(template="plotly_white")
    fig_lines.show()

# -------- C) Optional: quick peek of columns IF present (no KeyError) --------
peek_cols = [
    c for c in ["player", "season", "role", "rel_age", "spicy_weighted"] if c in zw.columns
]
if peek_cols:
    print(zw[peek_cols].head(8).to_string(index=False))
else:
    print("(Peek skipped: none of player/season/rel_age present.)")

       player season role rel_age  spicy_weighted
Adam Henrique  16-17    F      -1       -0.428285
Adam Henrique  19-20    F       2        0.931875
Adam Henrique  15-16    F      -2       -0.538525
Adam Henrique  17-18    F       0        0.683384
Adam Henrique  18-19    F       1       -0.648449
 Adam Larsson  20-21    D       0       -0.552449
 Adam Larsson  22-23    D       2        1.507333
 Adam Larsson  21-22    D       1        0.044183


In [59]:
# OLS: spicy_weighted ~ role * rel_age (categorical), HC3 robust SEs
if "rel_age" in zw.columns:
    m_sw = smf.ols("spicy_weighted ~ C(role) * C(rel_age, Treatment(0))", data=zw).fit(
        cov_type="HC3"
    )
else:
    m_sw = smf.ols("spicy_weighted ~ C(role)", data=zw).fit(cov_type="HC3")

print(m_sw.summary())

# Tidy
coefs_sw = pd.DataFrame(
    {
        "term": m_sw.params.index,
        "coef": m_sw.params.values,
        "se": m_sw.bse.values,
        "p": m_sw.pvalues.values,
    }
)
ci = m_sw.conf_int()
coefs_sw["ci_lo"] = ci[0].values
coefs_sw["ci_hi"] = ci[1].values
coefs_sw

                            OLS Regression Results                            
Dep. Variable:         spicy_weighted   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     1.835
Date:                Tue, 30 Sep 2025   Prob (F-statistic):             0.0580
Time:                        09:30:59   Log-Likelihood:                -1652.5
No. Observations:                1410   AIC:                             3325.
Df Residuals:                    1400   BIC:                             3377.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

,term,coef,se,p,ci_lo,ci_hi
0,Intercept,-0.013688,0.070520,0.846097,-0.151905,0.124529
1,C(role)[T.F],0.004032,0.091895,0.965007,-0.176079,0.184142
2,"C(rel_age, Treatment(0))[T.-2]",0.088182,0.112975,0.435070,-0.133245,0.309609
3,"C(rel_age, Treatment(0))[T.-1]",0.088050,0.100957,0.383129,-0.109823,0.285923
4,"C(rel_age, Treatment(0))[T.1]",-0.013579,0.108420,0.900331,-0.226078,0.198920
5,"C(rel_age, Treatment(0))[T.2]",-0.094213,0.105886,0.373599,-0.301746,0.113321
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.072322,0.141202,0.608519,-0.349074,0.204429
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.050575,0.129088,0.695216,-0.202432,0.303582
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.062279,0.134117,0.642388,-0.200585,0.325143
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.060689,0.136862,0.657452,-0.328934,0.207555


In [60]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px


# --- helpers ---
def season_end_from_str(s: str):
    if s is None:
        return pd.NA
    s = str(s).strip().replace("–", "-").replace("—", "-").replace("/", "-")
    m = re.fullmatch(r"(\d{2})-(\d{2})", s)
    if m:
        _, y2 = map(int, m.groups())
        return 2000 + y2
    m = re.fullmatch(r"(\d{4})-(\d{2,4})", s)
    if m:
        y1, y2 = m.groups()
        y1 = int(y1)
        y2 = int(y2) if len(y2) == 4 else (y1 // 100) * 100 + int(y2)
        if y2 < y1:
            y2 += 100
        return y2
    if re.fullmatch(r"\d{4}", s):
        return int(s)
    if re.fullmatch(r"\d{2}", s):
        return 2000 + int(s)
    return pd.NA


def norm_name(s):
    return " ".join(str(s).split()).strip().lower()


def ensure_cap_in_df(df):
    d = df.copy()
    # Case 1: already present
    if "_cap_hit" in d.columns:
        d["_cap_num"] = pd.to_numeric(d["_cap_hit"], errors="coerce")
        return d
    if "cap_hit_usd" in d.columns:
        d["_cap_num"] = pd.to_numeric(
            d["cap_hit_usd"].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
        )
        return d

    # Case 2: try to merge from aligned_cap CSV
    cap_path = Path("data/outputs/player_five_year_aligned_cap.csv")
    if not cap_path.exists():
        print(
            "⚠️ cap CSV not found at data/outputs/player_five_year_aligned_cap.csv and no cap columns in dfz; skipping chart."
        )
        return None

    cap = pd.read_csv(cap_path)
    # normalize keys
    player_col = (
        "player"
        if "player" in d.columns
        else ("player_name" if "player_name" in d.columns else None)
    )
    if player_col is None:
        # fall back: take any plausible name col
        for c in ["Player", "name", "Name"]:
            if c in d.columns:
                player_col = c
                break
    if player_col is None:
        print("⚠️ No player column in dfz to join caps; skipping chart.")
        return None

    d["_player_key"] = d[player_col].map(norm_name)
    # derive season_end in dfz
    if "season_end" in d.columns:
        d["_season_end"] = pd.to_numeric(d["season_end"], errors="coerce").astype("Int64")
    elif "season" in d.columns:
        d["_season_end"] = pd.to_numeric(
            d["season"].map(season_end_from_str), errors="coerce"
        ).astype("Int64")
    else:
        print("⚠️ Need season/season_end in dfz to join caps; skipping chart.")
        return None

    # cap table keys
    cap["_player_key"] = (
        cap["player"] if "player" in cap.columns else cap.get("player_name", cap.get("Player"))
    ).map(norm_name)
    if "season_end" in cap.columns:
        cap["_season_end"] = pd.to_numeric(cap["season_end"], errors="coerce").astype("Int64")
    elif "season" in cap.columns:
        cap["_season_end"] = pd.to_numeric(
            cap["season"].map(season_end_from_str), errors="coerce"
        ).astype("Int64")
    else:
        # try the script’s internal fields
        if "_season_end" not in cap.columns:
            print("⚠️ Cap CSV missing season key; skipping chart.")
            return None

    # prefer numeric _cap_hit; else parse cap_hit_usd
    if "_cap_hit" in cap.columns:
        cap["_cap_num"] = pd.to_numeric(cap["_cap_hit"], errors="coerce")
    elif "cap_hit_usd" in cap.columns:
        cap["_cap_num"] = pd.to_numeric(
            cap["cap_hit_usd"].astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
            errors="coerce",
        )
    else:
        print("⚠️ Cap CSV lacks _cap_hit/cap_hit_usd; skipping chart.")
        return None

    merged = d.merge(
        cap[["_player_key", "_season_end", "_cap_num"]].dropna(subset=["_cap_num"]),
        on=["_player_key", "_season_end"],
        how="left",
    )
    merged.drop(columns=["_player_key"], inplace=True, errors="ignore")
    return merged


def compute_peak_delta(df, dz_col="spicy_w_dz"):
    need = {"player", "role", "rel_age", dz_col}
    missing = need - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns for Peak Δ: {missing}")
    w = df[df["rel_age"].isin([-2, -1, 0, 1, 2])].copy()
    peak = w.groupby("player")[dz_col].mean().mul(-1).rename("peak_delta")
    role = w.groupby("player")["role"].first()
    return pd.concat([peak, role], axis=1).reset_index()


# --- prepare df with cap ---
dfw = ensure_cap_in_df(dfz)
if dfw is None:
    # bail gracefully without crashing the notebook
    pass
else:
    # rel_age tidy
    order = [-2, -1, 0, 1, 2]
    dfw["rel_age"] = pd.to_numeric(dfw["rel_age"], errors="coerce")
    dfw = dfw[dfw["rel_age"].isin(order)].copy()
    dfw["rel_age"] = pd.Categorical(dfw["rel_age"], categories=order, ordered=True)

    # attach Peak Δ and quintiles
    peak_w = compute_peak_delta(dfw, dz_col="spicy_w_dz")
    dfw = dfw.merge(peak_w[["player", "peak_delta", "role"]], on=["player", "role"], how="inner")
    dfw["pd_q"] = pd.qcut(dfw["peak_delta"], q=5, labels=[f"Q{i}" for i in range(1, 6)])

    # median cap by quantile × rel_age
    gcap = (
        dfw.groupby(["pd_q", "rel_age"], observed=True)["_cap_num"]
        .median()
        .reset_index(name="cap_median")
    )

    fig = px.line(
        gcap,
        x="rel_age",
        y="cap_median",
        color="pd_q",
        markers=True,
        category_orders={"rel_age": order, "pd_q": ["Q1", "Q2", "Q3", "Q4", "Q5"]},
        labels={"cap_median": "Median CapHit", "pd_q": "Peak Δ quintile"},
        title="Median CapHit by relative year, split by Peak Δ quintile (no ribbons)",
    )
    fig.update_traces(mode="lines+markers")
    fig.update_yaxes(tickformat=",.0f")
    fig.update_layout(template="plotly_white")
    fig.show()

⚠️ cap CSV not found at data/outputs/player_five_year_aligned_cap.csv and no cap columns in dfz; skipping chart.


In [61]:
# Build _cap_num from cap CSVs (no DB), merge into dfz, then plot median cap by Peak Δ quintile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px


# --- helpers ---
def season_end_from_str(s: str):
    if s is None:
        return pd.NA
    s = str(s).strip().replace("–", "-").replace("—", "-").replace("/", "-")
    m = re.fullmatch(r"(\d{2})-(\d{2})", s)
    if m:
        return 2000 + int(m.group(2))
    m = re.fullmatch(r"(\d{4})-(\d{2,4})", s)
    if m:
        y1, y2 = int(m.group(1)), m.group(2)
        y2 = int(y2) if len(y2) == 4 else (y1 // 100) * 100 + int(y2)
        if y2 < y1:
            y2 += 100
        return y2
    if re.fullmatch(r"\d{4}", s):
        return int(s)
    if re.fullmatch(r"\d{2}", s):
        return 2000 + int(s)
    return pd.NA


def norm_name(s):
    return " ".join(str(s).split()).strip().lower()


def compute_peak_delta(df, dz_col="spicy_w_dz"):
    need = {"player", "role", "rel_age", dz_col}
    missing = need - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns for Peak Δ: {missing}")
    w = df[df["rel_age"].isin([-2, -1, 0, 1, 2])].copy()
    peak = w.groupby("player")[dz_col].mean().mul(-1).rename("peak_delta")
    role = w.groupby("player")["role"].first()
    return pd.concat([peak, role], axis=1).reset_index()


# --- 1) Build cap table from data/cap_hits/*.csv ---
cap_dir = Path("data/cap_hits")
files = sorted(cap_dir.glob("player_cap_hits_*.csv"))
assert files, f"No cap files found in {cap_dir} (expected player_cap_hits_*.csv)."

caps = []
for f in files:
    m = re.search(r"(\d{4})", f.stem)
    if not m:
        continue
    yr = int(m.group(1))
    c = pd.read_csv(f, dtype=str, keep_default_na=False)
    name_col = "player name" if "player name" in c.columns else None
    if not name_col:
        raise SystemExit(f"{f}: missing 'player name' column.")
    cap_col = "capHit" if "capHit" in c.columns else ("cap_hit" if "cap_hit" in c.columns else None)
    if not cap_col:
        raise SystemExit(f"{f}: missing 'capHit' column.")
    c["_player_key"] = c[name_col].map(norm_name)
    c["_season_end"] = yr
    c["_cap_num"] = pd.to_numeric(
        c[cap_col].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
    )
    caps.append(c[["_player_key", "_season_end", "_cap_num"]])

cap_df = pd.concat(caps, ignore_index=True).drop_duplicates(
    ["_player_key", "_season_end"], keep="last"
)

# --- 2) Prepare keys in dfz and merge cap ---
d = dfz.copy()

# player key
player_col = (
    "player" if "player" in d.columns else ("player_name" if "player_name" in d.columns else None)
)
assert player_col, "dfz needs a player/player_name column."
d["_player_key"] = d[player_col].map(norm_name)

# season_end in dfz: prefer season_end; else parse season; else compute from peak_year + rel_age
if "season_end" in d.columns:
    d["_season_end"] = pd.to_numeric(d["season_end"], errors="coerce").astype("Int64")
elif "season" in d.columns:
    d["_season_end"] = pd.to_numeric(d["season"].map(season_end_from_str), errors="coerce").astype(
        "Int64"
    )
elif {"peak_year", "rel_age"}.issubset(d.columns):
    d["_season_end"] = pd.to_numeric(d["peak_year"], errors="coerce") + pd.to_numeric(
        d["rel_age"], errors="coerce"
    )
else:
    raise SystemExit(
        "dfz needs season_end OR season OR (peak_year + rel_age) to derive season_end."
    )

d = d.merge(cap_df, on=["_player_key", "_season_end"], how="left")

# --- 3) Tidy rel_age and compute Peak Δ (weighted) ---
order = [-2, -1, 0, 1, 2]
d["rel_age"] = pd.to_numeric(d["rel_age"], errors="coerce")
d = d[d["rel_age"].isin(order)].copy()
d["rel_age"] = pd.Categorical(d["rel_age"], categories=order, ordered=True)

peak_w = compute_peak_delta(d, dz_col="spicy_w_dz")
d = d.merge(peak_w[["player", "peak_delta", "role"]], on=["player", "role"], how="inner")

# quintiles of Peak Δ
d["pd_q"] = pd.qcut(d["peak_delta"], q=5, labels=[f"Q{i}" for i in range(1, 6)])

# --- 4) Chart: Median CapHit by rel_age within Peak Δ quintile ---
gcap = (
    d.groupby(["pd_q", "rel_age"], observed=True)["_cap_num"]
    .median()
    .reset_index(name="cap_median")
)

fig = px.line(
    gcap,
    x="rel_age",
    y="cap_median",
    color="pd_q",
    markers=True,
    category_orders={"rel_age": order, "pd_q": ["Q1", "Q2", "Q3", "Q4", "Q5"]},
    labels={"cap_median": "Median CapHit", "pd_q": "Peak Δ quintile"},
    title="Median CapHit by relative year, split by Peak Δ quintile (no ribbons)",
)
fig.update_traces(mode="lines+markers")
fig.update_yaxes(tickformat=",.0f")
fig.update_layout(template="plotly_white")
fig.show()

In [62]:
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

# Part A exports
raw[
    ["player", "season", "role", "cf_pct_num", "cf60_num", "ca60_num", "corsi_weighted_role"]
].to_csv(OUT / "role_weighted_corsi_raw.csv", index=False)
coefs_raw.to_csv(OUT / "reg_role_weighted_corsi_coefs.csv", index=False)

# Part B exports
coefs_spicy.to_csv(OUT / "reg_spicy_coefs.csv", index=False)

# Part C exports
coefs_sw.to_csv(OUT / "reg_spicy_weighted_coefs.csv", index=False)

print("Wrote exports to:", OUT.resolve())

NameError: name 'raw' is not defined

## Headline Numbers  This is the News! 
**Out of our sample, how many players show a peak in performance at age 27?**


In [64]:
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Inputs & checks ----
try:
    dfz = df_z.copy()
except NameError as e:
    raise NameError("df_z is not defined. Load the z-table first (see earlier cell).") from e

required_cols = {"player", "peak_year", "role", "rel_age", "cf60_dz", "ca60_dz", "cf_pct_dz"}
missing = required_cols - set(dfz.columns)
if missing:
    raise ValueError(f"Missing required columns in df_z: {sorted(missing)}")

# normalize role to {"D","F"}
dfz["role"] = np.where(dfz["role"].astype(str).str.upper().str.startswith("D"), "D", "F")
# enforce rel_age numeric and filter
VALID_RELS = [-2, -1, 0, 1, 2]
dfz["rel_age"] = pd.to_numeric(dfz["rel_age"], errors="coerce")
dfz = dfz[dfz["rel_age"].isin(VALID_RELS)].copy()


# ---- Part 1: Group means + 95% CIs by role × rel_age for each delta ----
def mean_ci(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    g = (
        df.dropna(subset=[value_col])
        .groupby(["role", "rel_age"], observed=True)
        .agg(n=(value_col, "size"), mean=(value_col, "mean"), sd=(value_col, "std"))
        .reset_index()
    )
    g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))
    g["ci_lo"] = g["mean"] - 1.96 * g["se"]
    g["ci_hi"] = g["mean"] + 1.96 * g["se"]
    g["metric"] = value_col
    return g


means_cf60 = mean_ci(dfz, "cf60_dz")
means_ca60 = mean_ci(dfz, "ca60_dz")
means_cfpc = mean_ci(dfz, "cf_pct_dz")

means_all = pd.concat([means_cf60, means_ca60, means_cfpc], ignore_index=True)

# quick peek
print("\n=== Group means (first 10 rows) ===")
print(means_all.head(10).to_string(index=False))


# ---- Part 2: Per-player concavity test & peak location in {-1,0,1} ----
# We fit y = a*x^2 + b*x + c on the 5 points (x = rel_age), then:
#   concave peak if a < 0 and vertex x* = -b/(2a) ∈ {-1,0,1} (allow small tolerance)
def concave_peak_in_window(
    x: np.ndarray, y: np.ndarray, window=(-1, 1), tol=0.35
) -> tuple[bool, float]:
    if len(x) < 5:
        return False, np.nan
    # Fit quadratic
    a, b, c = np.polyfit(x, y, deg=2)
    if a >= 0:
        return False, -b / (2 * a)  # not concave; still return vertex for info
    x_star = -b / (2 * a)
    in_window = (x_star >= window[0] - tol) and (x_star <= window[1] + tol)
    return in_window, x_star


def classify_metric(
    df: pd.DataFrame, value_col: str, invert_for_peak: bool = False
) -> pd.DataFrame:
    # Prepare long-form per (player, peak_year)
    cols = ["player", "peak_year", "role", "rel_age", value_col]
    sub = df[cols].dropna().copy()
    # Invert if needed (ca60_dz: smaller is better → maximize -ca60_dz)
    if invert_for_peak:
        sub["_y"] = -sub[value_col]
    else:
        sub["_y"] = sub[value_col]

    records = []
    for (pl, py), g in sub.groupby(["player", "peak_year"]):
        # Require all 5 rel_age points present
        if set(g["rel_age"]) >= set(VALID_RELS):
            x = g.sort_values("rel_age")["rel_age"].to_numpy()
            y = g.sort_values("rel_age")["_y"].to_numpy()
            ok, x_star = concave_peak_in_window(x, y, window=(-1, 1), tol=0.35)
            role = g["role"].mode(dropna=False).iat[0]
            records.append(
                {
                    "player": pl,
                    "peak_year": py,
                    "role": role,
                    "metric": value_col,
                    "concave_peak_-1to1": bool(ok),
                    "vertex_rel_age": float(x_star),
                }
            )
    return pd.DataFrame.from_records(records)


cls_cf60 = classify_metric(dfz, "cf60_dz", invert_for_peak=False)  # higher near 0 is better
cls_ca60 = classify_metric(dfz, "ca60_dz", invert_for_peak=True)  # invert: lower near 0 is better
cls_cfpc = classify_metric(dfz, "cf_pct_dz", invert_for_peak=False)

class_all = pd.concat([cls_cf60, cls_ca60, cls_cfpc], ignore_index=True)


# ---- Part 3: Headline counts (by metric × role and overall) ----
def headline_counts(df: pd.DataFrame, title: str):
    print(f"\n=== {title} ===")
    # by metric × role
    by_role = (
        df.groupby(["metric", "role"], observed=True)["concave_peak_-1to1"]
        .agg(n="size", n_peak="sum")
        .reset_index()
    )
    by_role["share_peak_%"] = 100 * by_role["n_peak"] / by_role["n"].replace(0, np.nan)
    print("\nBy metric × role:")
    print(by_role.to_string(index=False, formatters={"share_peak_%": lambda v: f"{v:5.1f}"}))

    # overall by metric
    overall = (
        df.groupby(["metric"], observed=True)["concave_peak_-1to1"]
        .agg(n="size", n_peak="sum")
        .reset_index()
    )
    overall["share_peak_%"] = 100 * overall["n_peak"] / overall["n"].replace(0, np.nan)
    print("\nOverall by metric:")
    print(overall.to_string(index=False, formatters={"share_peak_%": lambda v: f"{v:5.1f}"}))
    return by_role, overall


by_role_tbl, overall_tbl = headline_counts(class_all, "Concave peak in {-1,0,1} (per metric)")

# ---- Part 4: Save summary CSVs for the paper/notebook ----
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)
means_all.to_csv(OUT / "delta_means_ci_by_role_rel_age.csv", index=False)
class_all.to_csv(OUT / "concave_peak_classification_per_player.csv", index=False)

print("\nWrote:")
print(" -", OUT / "delta_means_ci_by_role_rel_age.csv")
print(" -", OUT / "concave_peak_classification_per_player.csv")

# ---- One-line “headline” you can quote in the writeup (example for CF60 only) ----
cf60_overall = overall_tbl[overall_tbl["metric"] == "cf60_dz"].iloc[0]
print(
    f"\nHeadline (CF60_dz): {cf60_overall['n_peak']}/{cf60_overall['n']} "
    f"({cf60_overall['share_peak_%']:.1f}%) players show a concave peak "
    f"in rel_age ∈ {{-1,0,1}}."
)


=== Group means (first 10 rows) ===
role  rel_age   n      mean       sd       se     ci_lo    ci_hi  metric
   D       -2 102 -0.038329 1.333723 0.132058 -0.297163 0.220505 cf60_dz
   D       -1 102 -0.068530 1.279906 0.126730 -0.316921 0.179860 cf60_dz
   D        0 102  0.000000 0.000000 0.000000  0.000000 0.000000 cf60_dz
   D        1 102  0.044570 1.303043 0.129020 -0.208310 0.297450 cf60_dz
   D        2 102 -0.033517 1.381589 0.136798 -0.301640 0.234607 cf60_dz
   F       -2 180 -0.041192 1.399357 0.104302 -0.245624 0.163240 cf60_dz
   F       -1 180  0.127508 1.320391 0.098416 -0.065388 0.320403 cf60_dz
   F        0 180  0.000000 0.000000 0.000000  0.000000 0.000000 cf60_dz
   F        1 180  0.085375 1.352324 0.100796 -0.112186 0.282935 cf60_dz
   F        2 180 -0.045781 1.499559 0.111771 -0.264851 0.173290 cf60_dz

=== Concave peak in {-1,0,1} (per metric) ===

By metric × role:
   metric role   n  n_peak share_peak_%
  ca60_dz    D 102      34         33.3
  ca60_dz    F

## Does Weighting the score really Tilt the Ice?


In [66]:
import numpy as np
import pandas as pd

# Expect df_z already loaded
dfz = df_z.copy()

# Ensure role is D/F
dfz["role"] = np.where(dfz["role"].astype(str).str.upper().str.startswith("D"), "D", "F")

# Sanity: required base z columns
base_needed = {"cf_pct_z", "cf60_z", "ca60_z"}
missing_base = base_needed - set(dfz.columns)
if missing_base:
    raise ValueError(f"df_z is missing base z columns: {sorted(missing_base)}")

# --- Unweighted composite z: mean(cf_pct_z, cf60_z, -ca60_z)
dfz["z_comp_unweighted"] = (dfz[["cf_pct_z", "cf60_z"]].mean(axis=1) - dfz["ca60_z"] / 2.0).where(
    ~dfz[["cf_pct_z", "cf60_z", "ca60_z"]].isna().any(axis=1), np.nan
)

# Note: The above equals (cf_pct_z + cf60_z - ca60_z)/3 scaled by 1.5.
# If you prefer strict mean, use:
# dfz["z_comp_unweighted"] = (dfz["cf_pct_z"] + dfz["cf60_z"] - dfz["ca60_z"]) / 3.0

# --- Role-weighted composite z (heuristics)
# D: 0.5*cf_pct_z + 0.2*cf60_z - 0.3*ca60_z
# F: 0.5*cf_pct_z + 0.3*cf60_z - 0.2*ca60_z
w_cf60 = np.where(dfz["role"] == "D", 0.2, 0.3)
w_ca60 = np.where(dfz["role"] == "D", 0.3, 0.2)
dfz["z_comp_weighted"] = 0.5 * dfz["cf_pct_z"] + w_cf60 * dfz["cf60_z"] - w_ca60 * dfz["ca60_z"]

# Optional: if spicy_weighted not present, create it from spicy components
if "spicy_score" in dfz.columns and "spicy_weighted" not in dfz.columns:
    # Equal-weight spicy (already in spicy_score) ~ mean of cf_pct_z, cf60_z, -ca60_z
    # Weighted spicy mirrors composite weights:
    dfz["spicy_weighted"] = 0.5 * dfz["cf_pct_z"] + w_cf60 * dfz["cf60_z"] - w_ca60 * dfz["ca60_z"]

# Put back for downstream cells
df_z = dfz

print(
    "Added columns:",
    [c for c in ["z_comp_unweighted", "z_comp_weighted", "spicy_weighted"] if c in df_z.columns],
)
print(df_z[["role", "rel_age", "z_comp_unweighted", "z_comp_weighted"]].head())

Added columns: ['z_comp_unweighted', 'z_comp_weighted', 'spicy_weighted']
  role rel_age  z_comp_unweighted  z_comp_weighted
0    F      -1          -0.608991        -0.428285
1    F       2           1.168489         0.931875
2    F      -2          -0.348950        -0.538525
3    F       0           0.780327         0.683384
4    F       1          -0.990875        -0.648449


In [67]:
# === Does weighting matter? (Composite Corsi z and Spicy) ===
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

# Expect df_z already loaded. Check required columns.
need = {
    "player",
    "peak_year",
    "role",
    "rel_age",
    "z_comp_unweighted",
    "z_comp_weighted",
    "spicy_score",
    "spicy_weighted",
}
missing = need - set(df_z.columns)
if missing:
    raise ValueError(f"df_z is missing columns: {sorted(missing)}")

z = df_z.copy()
# Normalize role and rel_age
z["role"] = np.where(z["role"].astype(str).str.upper().str.startswith("D"), "D", "F")
order = [-2, -1, 0, 1, 2]
z["rel_age"] = pd.Categorical(
    pd.to_numeric(z["rel_age"], errors="coerce"), categories=order, ordered=True
)


def mean_ci(df, col, by=("role", "rel_age")):
    g = (
        df.dropna(subset=[col])
        .groupby(list(by), observed=True)
        .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
        .reset_index()
    )
    g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))
    g["ci_lo"] = g["mean"] - 1.96 * g["se"]
    g["ci_hi"] = g["mean"] + 1.96 * g["se"]
    g["measure"] = col
    return g


def summarize_diffs(df, weighted_col, unweighted_col, label):
    d = df[["player", "peak_year", "role", "rel_age", weighted_col, unweighted_col]].dropna().copy()
    d["diff"] = d[weighted_col] - d[unweighted_col]
    d["family"] = label
    return d


# 1) Build diff frames
d_corsi = summarize_diffs(z, "z_comp_weighted", "z_comp_unweighted", "Composite Corsi z")
d_spicy = summarize_diffs(z, "spicy_weighted", "spicy_score", "Spicy z")

diffs = pd.concat([d_corsi, d_spicy], ignore_index=True)

# 2) Summaries by role × rel_age
sum_corsi = mean_ci(d_corsi, "diff", by=("role", "rel_age"))
sum_spicy = mean_ci(d_spicy, "diff", by=("role", "rel_age"))

print("=== Mean difference (weighted − unweighted) by role × rel_age ===")
print("\nComposite Corsi z:")
print(sum_corsi.to_string(index=False, float_format=lambda v: f"{v: .3f}"))
print("\nSpicy z:")
print(sum_spicy.to_string(index=False, float_format=lambda v: f"{v: .3f}"))


# 3) Quick tests: is mean(diff) != 0 ? (overall and by role)
def mean_test_table(df, label):
    tbl = (
        df.groupby("role", observed=True)["diff"].agg(n="size", mean="mean", sd="std").reset_index()
    )
    # overall row
    overall = df["diff"].agg(n="size", mean="mean", sd="std").to_dict()
    overall["role"] = "ALL"
    tbl = pd.concat([tbl, pd.DataFrame([overall])], ignore_index=True)

    # Normal approx p-value (two-sided). For large n this is ok; else swap in scipy if desired.
    tbl["se"] = tbl["sd"] / np.sqrt(tbl["n"].clip(lower=1))
    tbl["z"] = tbl["mean"] / tbl["se"].replace(0, np.nan)
    # 95% CI
    tbl["ci_lo"] = tbl["mean"] - 1.96 * tbl["se"]
    tbl["ci_hi"] = tbl["mean"] + 1.96 * tbl["se"]

    print(f"\n=== Mean(diff) test for {label} (weighted − unweighted) ===")
    print(
        tbl[["role", "n", "mean", "se", "ci_lo", "ci_hi", "z"]].to_string(
            index=False, float_format=lambda v: f"{v: .4f}"
        )
    )
    return tbl


test_corsi = mean_test_table(d_corsi, "Composite Corsi z")
test_spicy = mean_test_table(d_spicy, "Spicy z")

# 4) Visuals — lines & histograms


# (A) Line: means of weighted vs unweighted by rel_age (separate traces) for each role
def long_two(df, a, b, label_a, label_b, family_label):
    t = (
        df[["role", "rel_age", a, b]]
        .dropna()
        .melt(id_vars=["role", "rel_age"], value_vars=[a, b], var_name="which", value_name="value")
    )
    t["which"] = t["which"].map({a: label_a, b: label_b})
    t["family"] = family_label
    return t


long_corsi = long_two(
    z, "z_comp_unweighted", "z_comp_weighted", "unweighted", "weighted", "Composite Corsi z"
)
long_spicy = long_two(z, "spicy_score", "spicy_weighted", "unweighted", "weighted", "Spicy z")

for fam, df_long in [("Composite Corsi z", long_corsi), ("Spicy z", long_spicy)]:
    fig = px.line(
        df_long,
        x="rel_age",
        y="value",
        color="which",
        facet_col="role",
        category_orders={"rel_age": order},
        markers=True,
        title=f"{fam}: weighted vs unweighted (by rel_age, faceted by role)",
        labels={"value": fam, "which": "series"},
    )
    fig.update_layout(height=380)
    fig.show()

# (B) Hist: distribution of (weighted − unweighted) by role
for fam, d in [("Composite Corsi z", d_corsi), ("Spicy z", d_spicy)]:
    fig = px.histogram(
        d,
        x="diff",
        color="role",
        barmode="overlay",
        nbins=50,
        opacity=0.55,
        title=f"{fam}: distribution of (weighted − unweighted) by role",
        labels={"diff": "weighted − unweighted"},
    )
    fig.show()


# 5) Correlation between weighted & unweighted (how similar are the rankings?)
def corr_table(df, a, b, name):
    cc = (
        df[[a, b, "role"]]
        .dropna()
        .groupby("role", observed=True)
        .apply(lambda g: pd.Series({"pearson_r": g[a].corr(g[b])}))
        .reset_index()
    )
    # overall
    overall_r = df[[a, b]].dropna().corr().iloc[0, 1]
    cc = pd.concat([cc, pd.DataFrame([{"role": "ALL", "pearson_r": overall_r}])], ignore_index=True)
    print(f"\n=== Correlation {name} (weighted vs unweighted) ===")
    print(cc.to_string(index=False, float_format=lambda v: f"{v: .3f}"))
    return cc


corr_corsi = corr_table(z, "z_comp_unweighted", "z_comp_weighted", "Composite Corsi z")
corr_spicy = corr_table(z, "spicy_score", "spicy_weighted", "Spicy z")

# 6) Save small summaries for the paper
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)
sum_corsi.assign(family="Composite Corsi z").to_csv(
    OUT / "weighting_effect_corsi_by_role_rel_age.csv", index=False
)
sum_spicy.assign(family="Spicy z").to_csv(
    OUT / "weighting_effect_spicy_by_role_rel_age.csv", index=False
)
test_corsi.assign(family="Composite Corsi z").to_csv(OUT / "weighting_tests_corsi.csv", index=False)
test_spicy.assign(family="Spicy z").to_csv(OUT / "weighting_tests_spicy.csv", index=False)
corr_corsi.assign(family="Composite Corsi z").to_csv(OUT / "weighting_corr_corsi.csv", index=False)
corr_spicy.assign(family="Spicy z").to_csv(OUT / "weighting_corr_spicy.csv", index=False)

print("\nSaved CSVs in data/outputs/")

=== Mean difference (weighted − unweighted) by role × rel_age ===

Composite Corsi z:
role rel_age   n   mean     sd     se  ci_lo  ci_hi measure
   D      -2 102 -0.025  0.397  0.039 -0.102  0.052    diff
   D      -1 102 -0.016  0.338  0.033 -0.081  0.050    diff
   D       0 102  0.000  0.314  0.031 -0.061  0.061    diff
   D       1 102 -0.001  0.375  0.037 -0.074  0.072    diff
   D       2 102  0.042  0.354  0.035 -0.026  0.111    diff
   F      -2 180 -0.006  0.363  0.027 -0.059  0.047    diff
   F      -1 180 -0.056  0.327  0.024 -0.103 -0.008    diff
   F       0 180 -0.011  0.359  0.027 -0.063  0.042    diff
   F       1 180 -0.008  0.305  0.023 -0.053  0.036    diff
   F       2 180  0.081  0.377  0.028  0.026  0.136    diff

Spicy z:
role rel_age   n   mean     sd     se  ci_lo  ci_hi measure
   D      -2 102  0.008  0.077  0.008 -0.007  0.023    diff
   D      -1 102  0.014  0.069  0.007  0.001  0.028    diff
   D       0 102 -0.005  0.071  0.007 -0.018  0.009    diff
   D


=== Correlation Composite Corsi z (weighted vs unweighted) ===
role  pearson_r
   D      0.996
   F      0.996
 ALL      0.996

=== Correlation Spicy z (weighted vs unweighted) ===
role  pearson_r
   D      0.996
   F      0.996
 ALL      0.996

Saved CSVs in data/outputs/


/var/folders/hb/kbyr_0y166n0nvnd_cqyjzth0000gn/T/ipykernel_43350/742540563.py:126: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/hb/kbyr_0y166n0nvnd_cqyjzth0000gn/T/ipykernel_43350/742540563.py:126: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.





## Weighted vs. Unweighted — What changed?

**Bottom line:** Almost nothing. Across Defense and Forwards, the weighted and unweighted composites are *virtually interchangeable*.

### Evidence

* **Correlations:** r ≈ **0.996** (D, F, and overall) between weighted and unweighted composite z’s → the two scores move together almost perfectly.
* **Mean differences (weighted − unweighted):**

  * By role × rel_age: all means are very small (mostly in **±0.06 z**), with CIs that overwhelmingly **span 0**.

    * One small blip: **Forwards @ rel_age = −1** shows −0.056 (95% CI **[−0.103, −0.008]**), i.e., a tiny downward shift when weighting—directionally consistent with giving F a bit more creation and a bit less suppression weight—but **still very small** in magnitude.
  * **Overall (pooled by role):** mean diffs are essentially **0.000** with tight CIs (e.g., ALL: 95% CI **[−0.018, 0.018]** for composite Corsi; **[−0.004, 0.004]** for spicy).

### Interpretation

* The heuristic weights (D: 0.5/0.2/−0.3 vs F: 0.5/0.3/−0.2 on cf_pct_z/cf60_z/ca60_z) **do not materially change** rankings or conclusions.
* Any “shifts” are **tiny**—on the order of **hundredths of a z-score**, i.e., practically negligible at the player-season level.
* For this milestone, it’s reasonable to present **unweighted** composite results as the primary view and note that the **weighted** variant yields the **same story**.

### Slide one-liner

> **Weighted vs. Unweighted:** No meaningful difference — r=0.996 and mean(delta)≈0 with CIs spanning 0. Same conclusions either way.

## Why the weighted band is tighter

* The **weighted composite** (role-aware z) gives **less influence** to noisier pieces (e.g., creation for D, suppression for F).
* That **reduces variance** without meaningfully shifting the center, so the **95% CI ribbon is narrower** and often **fits inside** the unweighted ribbon.
* In other words, weighting acts like a mild **regularizer**: it **stabilizes** the composite but **doesn’t change the story**—your weighted and unweighted curves are almost identical in level (correlations ≈ 1), with the weighted one simply **less wobbly**.



## Fun Bonus Awesome—let’s quantify PeakΔ (“how far the peak season sits above the 5-year norm”) cleanly and plot it.

In [71]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

VALID_RELS = [-2, -1, 0, 1, 2]


def compute_peak_delta(df, value_col, group_cols=("player", "peak_year"), role_col="role"):
    """
    Returns a DataFrame with one row per (player, peak_year[, role]) containing:
        - peak_val (mean at rel_age=0)
        - win_mean (mean over rel_age in {-2,-1,0,1,2})
        - peak_delta = peak_val - win_mean
    """
    need = set(group_cols) | {"rel_age", value_col}
    missing = need - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns for PeakΔ: {sorted(missing)}")

    dd = df.copy()
    dd["rel_age"] = pd.to_numeric(dd["rel_age"], errors="coerce")
    dd = dd[dd["rel_age"].isin(VALID_RELS)].dropna(subset=[value_col])

    # mean at peak (rel_age = 0)
    pk = (
        dd.loc[dd["rel_age"].eq(0)]
        .groupby(list(group_cols), as_index=False)[value_col]
        .mean()
        .rename(columns={value_col: "peak_val"})
    )

    # mean across 5-year window
    win = (
        dd.groupby(list(group_cols), as_index=False)[value_col]
        .mean()
        .rename(columns={value_col: "win_mean"})
    )

    out = pk.merge(win, on=list(group_cols), how="inner")
    out["peak_delta"] = out["peak_val"] - out["win_mean"]

    # carry role (optional) for coloring; best-effort merge
    if role_col in df.columns:
        roles = df.groupby(list(group_cols), as_index=False)[role_col].agg(
            lambda s: s.dropna().iloc[0] if len(s.dropna()) else None
        )
        out = out.merge(roles, on=list(group_cols), how="left")

    return out


def bar_topk_peak_delta(peak_df, k=5, title="Peak Δ teaser (Top 5) — higher = bigger lift vs norm"):
    topk = peak_df.nlargest(min(k, len(peak_df)), "peak_delta").copy()
    # Label with player and (optional) year
    ylabels = topk["player"].astype(str) + np.where(
        "peak_year" in topk.columns, " (" + topk["peak_year"].astype(str) + ")", ""
    )

    palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    colors = [palette.get(r, "#888") for r in topk.get("role", pd.Series([None] * len(topk)))]

    fig = go.Figure(
        go.Bar(
            x=topk["peak_delta"],
            y=ylabels,
            orientation="h",
            marker_color=colors,
            customdata=topk.get("role", pd.Series(["?"] * len(topk))),
            hovertemplate="<b>%{y}</b><br>Peak Δ: %{x:.3f}<br>Role: %{customdata}<extra></extra>",
        )
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_title="Peak Δ",
        yaxis_title="",
        height=320,
        margin=dict(l=140, r=10, t=50, b=30),
        showlegend=False,
    )
    fig.update_yaxes(autorange="reversed")
    return fig


def hist_peak_delta(peak_df, title="Peak Δ — distribution by role"):
    fig = px.histogram(
        peak_df,
        x="peak_delta",
        color="role",
        barmode="overlay",
        nbins=40,
        opacity=0.6,
        template="plotly_white",
        title=title,
        labels={"peak_delta": "Peak Δ"},
    )
    return fig

In [72]:
# Equal-weight (spicy_score) and role-weighted (spicy_weighted)
pd_eq = compute_peak_delta(dfz, "spicy_score")
pd_w = compute_peak_delta(dfz, "spicy_weighted")

# Quick sanity
print("Rows (equal-weight):", len(pd_eq), " | Rows (weighted):", len(pd_w))

# Top-5 teasers
bar_topk_peak_delta(pd_eq, k=5, title="Peak Δ (Equal-weight z) — Top 5").show()
bar_topk_peak_delta(pd_w, k=5, title="Peak Δ (Role-weighted z) — Top 5").show()

# Distributions
hist_peak_delta(pd_eq, "Peak Δ (Equal-weight z) — distribution by role").show()
hist_peak_delta(pd_w, "Peak Δ (Role-weighted z) — distribution by role").show()

# Compare the two peak deltas (correlation + small table)
cmp = pd_eq.merge(pd_w, on=["player", "peak_year", "role"], suffixes=("_eq", "_w"))
r = cmp["peak_delta_eq"].corr(cmp["peak_delta_w"])
print(f"Correlation PeakΔ (equal vs weighted): {r:.3f}")

cmp[["player", "peak_year", "role", "peak_delta_eq", "peak_delta_w"]].head(10)

Rows (equal-weight): 282  | Rows (weighted): 282


Correlation PeakΔ (equal vs weighted): 0.996


,player,peak_year,role,peak_delta_eq,peak_delta_w
0,Adam Henrique,2018,F,0.520218,0.683384
1,Adam Larsson,2021,D,-0.557387,-0.552449
2,Adam Lowry,2021,F,-0.020980,-0.167344
3,Adam Pelech,2022,D,0.219044,0.252817
4,Aleksander Barkov,2023,F,-1.183246,-1.286608
5,Alex Chiasson,2019,F,-0.499930,-0.468357
6,Alex Iafallo,2022,F,1.391969,1.445091
7,Alex Kerfoot,2022,F,-0.029342,-0.002588
8,Alex Killorn,2017,F,-1.338455,-1.381993
9,Alex Pietrangelo,2018,D,-0.271961,-0.395519
